# Calibración de la voz del narrador

Este notebook convierte **tu grabación** en la voz de referencia del canal y prueba varias configuraciones de Chatterbox Multilingual con 10 frases técnicas (§10.3 del plan).

Al terminar tendrás:
- `assets/voz_referencia/narrador.wav`: la referencia limpia que usará el pipeline.
- `build/calibracion_voz/`: los audios de prueba, con su WER medido con Whisper.
- `config/voz.yaml`: actualizado con la configuración que elijas.

Se ejecuta con el kernel `series-redes-neuronales/.venv` (Python 3.13). La primera vez descarga Chatterbox (~3 GB) y Whisper large-v3 (~3 GB).

## 1. Graba tu voz

Chatterbox **solo usa los primeros 10 segundos** de la referencia (y de ellos, 6 s para el estilo). Imita el timbre, el ritmo y la energía, así que graba con el tono exacto que quieres para el narrador.

- **Duración:** 15–20 s. Empieza a hablar enseguida; el notebook recorta el silencio inicial.
- **Sala:** silenciosa y sin eco (un armario con ropa o una habitación con cortinas funciona bien). Nada de música ni ventiladores.
- **Micrófono:** a 15–25 cm de la boca, sin saturar. Un móvil moderno sirve.
- **Tono:** cercano y claro, a ritmo de explicación, no de lectura rápida.
- **Formato:** cualquiera que lea ffmpeg (wav, m4a, mp3…). Graba 2–3 tomas y guárdalas en `assets/voz_referencia/grabaciones/`.

Texto sugerido (tiene preguntas, pausas y variedad de sonidos):

> Hola, bienvenido. Hoy vamos a entender cómo aprende una red neuronal, paso a paso y sin prisas. Primero veremos la intuición; después, solo las matemáticas justas. ¿Preparado? Empezamos con algo muy sencillo: una sola neurona.

In [ ]:
import gc, hashlib, json, subprocess, sys, time, unicodedata, re
from datetime import date
from pathlib import Path

RAIZ = Path.cwd().resolve()
while not (RAIZ / "CLAUDE.md").exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

import pandas as pd
import torch
import soundfile as sf
import yaml
from IPython.display import Audio, HTML, display

from lib import config
from lib.texto import texto_para_tts, terminos_pronunciacion

print(RAIZ, "·", torch.__version__, "·", torch.cuda.get_device_name(0))

## 2. Configuración de la prueba

In [ ]:
# Tu grabación (cambia el nombre para probar otra toma).
TOMA = RAIZ / "assets/voz_referencia/grabaciones/toma2.m4a"
REFERENCIA = RAIZ / "assets/voz_referencia/narrador.wav"
SALIDA = RAIZ / "build/calibracion_voz" / TOMA.stem   # una carpeta por toma: nunca se mezclan voces
SEMILLA = 42

FRASES = [
    "Una red neuronal es una composición de funciones sencillas que deforma el espacio.",
    "Con backpropagation, el gradiente fluye hacia atrás por cada capa de la red.",
    "La función de activación ReLU deja pasar los valores positivos y anula los negativos.",
    "Entrenamos el modelo en PyTorch sobre un dataset público de dígitos escritos a mano.",
    "La LSTM decide qué recordar y qué olvidar gracias a sus puertas.",
    "En un Transformer, cada palabra decide a qué otras palabras prestar atención.",
    "El softmax convierte las puntuaciones en probabilidades que suman uno.",
    "Una GAN enfrenta a dos redes: un falsificador y un detective.",
    "Cada nodo de una GNN aprende escuchando a sus vecinos.",
    "Con una tasa de aprendizaje de 0,001, la pérdida baja de 2,3 a 0,4 en diez épocas.",
]

# A = valores iniciales del plan; B = valores por defecto de Chatterbox;
# C = más calmada y pausada; D = más expresiva.
CONFIGS = {
    "A": dict(exaggeration=0.4, cfg_weight=0.5, temperature=0.7),
    "B": dict(exaggeration=0.5, cfg_weight=0.5, temperature=0.8),
    "C": dict(exaggeration=0.3, cfg_weight=0.3, temperature=0.7),
    "D": dict(exaggeration=0.6, cfg_weight=0.4, temperature=0.75),
}

## 3. Preparar la referencia

Recorta silencios al principio y al final, quita graves de ruido (<70 Hz), pasa a mono 48 kHz y normaliza la sonoridad. Escucha el resultado: si suena con eco o ruido, repite la grabación antes de seguir.

In [ ]:
REFERENCIA.parent.mkdir(parents=True, exist_ok=True)
recorte = "silenceremove=start_periods=1:start_threshold=-45dB:start_silence=0.05"
filtro = f"{recorte},areverse,{recorte},areverse,highpass=f=70,loudnorm=I=-20:TP=-2:LRA=11"
subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", str(TOMA), "-af", filtro,
                "-ac", "1", "-ar", "48000", str(REFERENCIA)], check=True)

duracion_ref = sf.info(str(REFERENCIA)).duration
print(f"Referencia: {duracion_ref:.1f} s")
if duracion_ref < 10:
    print("⚠ Menos de 10 s: Chatterbox aprovecha hasta 10 s, graba algo más largo.")
display(Audio(str(REFERENCIA)))

## 4. Generar las frases de prueba

10 frases × 4 configuraciones = 40 audios. Con la GPU tarda unos minutos. Los audios ya generados de **esta toma** se reutilizan (cada toma tiene su carpeta); `REGENERAR = True` fuerza a generarlos otra vez.

In [ ]:
from chatterbox.mtl_tts import ChatterboxMultilingualTTS

REGENERAR = False
t0 = time.time()
modelo = ChatterboxMultilingualTTS.from_pretrained(device="cuda")
print(f"Modelo cargado en {time.time() - t0:.0f} s · sr={modelo.sr}")

filas = []
for nombre, params in CONFIGS.items():
    carpeta = SALIDA / nombre
    carpeta.mkdir(parents=True, exist_ok=True)
    condiciones_listas = False
    for i, frase in enumerate(FRASES, 1):
        ruta = carpeta / f"f{i:02d}.wav"
        t0 = time.time()
        if REGENERAR or not ruta.exists():
            if not condiciones_listas:   # una vez por configuración: exaggeration afecta a las condiciones
                modelo.prepare_conditionals(str(REFERENCIA), exaggeration=params["exaggeration"])
                condiciones_listas = True
            torch.manual_seed(SEMILLA)
            wav = modelo.generate(texto_para_tts(frase), language_id="es", **params)
            sf.write(str(ruta), wav.squeeze(0).cpu().numpy(), modelo.sr)
        dur = sf.info(str(ruta)).duration
        filas.append(dict(config=nombre, frase=i, ruta=ruta, duracion=dur, s_por_caracter=dur / len(frase),
                          seg_generacion=time.time() - t0))
    print(f"{nombre} listo")

gen = pd.DataFrame(filas)
del modelo; gc.collect(); torch.cuda.empty_cache()   # libera la VRAM antes de cargar Whisper

## 5. QA con Whisper

Transcribe cada audio y calcula el WER frente al texto original, **excluyendo los términos del diccionario de pronunciación** (Whisper los escribe a su manera) y quitando tildes y puntuación. Los números en cifras cuentan como error si Whisper los escribe en letras o al revés; fíjate en la frase 10 al escucharla más que en su WER.

In [ ]:
import jiwer
from faster_whisper import WhisperModel

qa = config.cargar("pipeline")["voz_qa"]
EXCLUIR = terminos_pronunciacion()

def normalizar(texto: str) -> str:
    texto = unicodedata.normalize("NFD", texto.lower())
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    palabras = re.sub(r"[^\w\s]", " ", texto).split()
    return " ".join(p for p in palabras if p not in EXCLUIR)

whisper = WhisperModel(qa["modelo_whisper"], device="cuda", compute_type="float16")
transcripciones, wers = [], []
for fila in gen.itertuples():
    segmentos, _ = whisper.transcribe(str(fila.ruta), language="es", beam_size=5)
    texto = " ".join(s.text.strip() for s in segmentos)
    transcripciones.append(texto)
    wers.append(jiwer.wer(normalizar(FRASES[fila.frase - 1]), normalizar(texto)))
gen["transcripcion"] = transcripciones
gen["wer"] = wers
del whisper; gc.collect(); torch.cuda.empty_cache()

lo, hi = qa["seg_por_caracter"]
gen["duracion_ok"] = gen.s_por_caracter.between(lo, hi)
resumen = gen.groupby("config").agg(
    wer_medio=("wer", "mean"), wer_max=("wer", "max"),
    frases_sobre_umbral=("wer", lambda w: int((w > qa["wer_max_frase"]).sum())),
    s_por_caracter=("s_por_caracter", "mean"), duracion_fuera_rango=("duracion_ok", lambda d: int((~d).sum())),
    seg_generacion_medio=("seg_generacion", "mean"),
).round(3)
display(resumen)

In [ ]:
# Detalle: frases con WER alto o duración anómala (candidatas a nuevas entradas en pronunciacion.yaml).
problemas = gen[(gen.wer > qa["wer_max_frase"]) | ~gen.duracion_ok]
display(problemas[["config", "frase", "wer", "s_por_caracter", "transcripcion"]] if len(problemas) else "Sin problemas 🎉")

## 6. Escuchar y comparar

El WER dice si se entiende; **el oído decide si suena natural**. Escucha al menos las frases 2, 3, 4 y 10 en todas las configuraciones.

In [ ]:
for i, frase in enumerate(FRASES, 1):
    display(HTML(f"<h4>{i}. {frase}</h4>"))
    for fila in gen[gen.frase == i].itertuples():
        print(f"{fila.config}  WER={fila.wer:.2f}  «{fila.transcripcion}»")
        display(Audio(str(fila.ruta)))

## 7. Fijar la configuración elegida

Escribe la letra de la que más te guste. Se guarda en `config/voz.yaml` junto con la huella de la referencia, para detectar si alguien cambia `narrador.wav` sin recalibrar.

In [ ]:
ELEGIDA = "A"   # ← cambia por la configuración que prefieras

ruta_voz = RAIZ / "config/voz.yaml"
voz = yaml.safe_load(ruta_voz.read_text(encoding="utf-8"))
voz["parametros"] = CONFIGS[ELEGIDA]
voz["semilla"] = SEMILLA
voz["calibracion"] = {
    "fecha": date.today().isoformat(),
    "toma": str(TOMA.relative_to(RAIZ)),
    "sha256_referencia": hashlib.sha256(REFERENCIA.read_bytes()).hexdigest(),
    "wer_medio": float(resumen.loc[ELEGIDA, "wer_medio"]),
}
cabecera = "# Configuración del narrador (§10.3). Generado por notebooks/01_calibrar_voz.ipynb.\n"
ruta_voz.write_text(cabecera + yaml.safe_dump(voz, allow_unicode=True, sort_keys=False), encoding="utf-8")
print(ruta_voz.read_text(encoding="utf-8"))